## §1 — Libraries

In [ ]:
%run _bootstrap_dev.ipynb

## §2 — Portafoglio da analizzare

In [ ]:
#
# Quale portafoglio analizzare
#
# portfolio = portfolio_alpha_euro
# portfolio = portfolio_alpha_sect
# portfolio = portfolio_alpha_world
# portfolio = portfolio_alpha_world_vanguard
# portfolio = portfolio_alpha_nasdaq100
# portfolio = portfolio_alpha_sp100
portfolio = portfolio_germany_plan
# portfolio = portfolio_italy_big_cap
# portfolio = portfolio_alpha_quant # New Ema
# portfolio = portfolio_alpha_fact

# Profile di destinazione del PTF nel portafoglio del cliente
# Determina le soglie di overfitting check (S1-S4).
# "satellite": quota tattica, cerca alpha; "core": quota di base, prioritizza capital preservation.
profile = "satellite"   # "satellite" | "core"
pf_rot_cluster = None 
pf_rot_cluster_base = None

# --- Derivazione automatica dalle proprieta' del portfolio ---
# Flag calcolato PRIMA della risoluzione Wikipedia, quando tickers è ancora stringa.
survivorship_bias_universe = isinstance(portfolio['tickers'], str)
tickers = portfolio['tickers']

tickers = (
    extract_tickers_from_wikipedia(tickers, exclude=["GOOG"], rename={"BRK.B": "BRK-B"})
    if isinstance(tickers, str)
    else list(tickers)
)
benchmark_portfolio = portfolio['benchmark_portfolio']
benchmark_title     = portfolio['benchmark_title']
portfolio_title     = portfolio['Title']

# Percorso file WFO
wfo_results_dir = _TSLAB_DEV_R_WFO_RESULTS_DIR
wfo_file_save   = f"{wfo_results_dir}/{portfolio_title}_{year}.wfo_summary.csv"

# Date di analisi
start_date = "2015-01-01"
end_date   = None          # None = oggi
year       = 2026          # anno di selezione corrente

print(f"Portafoglio: {BOLD}{portfolio_title}{RESET}  |  Profile: {BOLD}{profile}{RESET}")
print(f"Ticker: {len(tickers)}  |  Benchmark: {benchmark_title}")
print(f"WFO file: {wfo_file_save}")
print(f"Survivorship bias universe: {survivorship_bias_universe}")

reports_dir = get_analysis_output_dir("r_analysis", ptf_name=portfolio_title.replace(' ', '_').lower(), profilo=profile)
plots_dir   = reports_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"reports_dir: {reports_dir}")
print(f"plots_dir: {plots_dir}")

### §2a — Load WFO Results

Se eseguita si puo' saltare a §4a (Confronto WFO) perche' carica i risultati di una run precedente
(da notebook o da `iq r-analyze`) senza rieseguire §4.

**Scope**: copre solo §4a. Per proseguire con §4b occorre eseguire  §3 — Download data

In [ ]:
# ptf_name e profile sono già definiti in §1
# Per caricare una run specifica: wfo_resume = load_wfo_results_for_resume(ptf_name, profile, run_dir="path/to/run")

wfo_resume = load_wfo_results_for_resume(ptf_name=portfolio_title, profilo=profile)

# Ripristina variabili di sessione per §4a
results_pipeline       = wfo_resume["results_pipeline"]

pipeline_start_date    = wfo_resume["pipeline_start_date"]
ratio                  = wfo_resume["ratio"]
metric                 = wfo_resume["metric"]
force_next_year_params = wfo_resume["force_next_year_params"]
autoreduce             = wfo_resume["autoreduce"]
risk_on_off            = wfo_resume["risk_on_off"]

print(f"\nResume caricato: {wfo_resume['run_dir']}")
print(f"Anno WFO: {wfo_resume['year']}  |  Engines: {list(results_pipeline.keys())}")

## §3 — Download data

In [ ]:
init_cash      = 100_000
normalize      = False    # lasciare False nei rotazionali: NaN = titolo assente
lookback_buffer = 365

download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")

stocks_data, stocks_data_raw, company_data = fetch_data_adjusted_and_raw(
    tickers, download_start_date, end_date, normalize=normalize
)

# Usato dalla Stability Analysis 
portfolio["stocks_data"] = stocks_data
portfolio["init_cash"]   = init_cash

if benchmark_portfolio:
    benchmark_data     = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max()).replace(0, np.nan).ffill()
    benchmark_data_raw = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max(),
                             auto_adjust=False).replace(0, np.nan).ffill()
elif benchmark_title:
    benchmark_data, benchmark_data_raw = fetch_series_adjusted_and_raw(
        benchmark_title, stocks_data.index.min(), end_date
    )
else:
    benchmark_data = benchmark_data_raw = None

print (f"Portfolio {BOLD}{portfolio_title}{RESET} tickers:")
display(stocks_data)
risk_off_tickers = portfolio.get('risk_off_tickers', risk_off_tickers)
risk_off_tickers_uniq = [
    t for t in risk_off_tickers
    if t not in tickers
]

# Tickers da usare in modalita risk_off (difensivi)
risk_off_data = download_data(risk_off_tickers_uniq, download_start_date, end_date) if risk_off_tickers else None
# workaround: forza sempre DataFrame
if isinstance(risk_off_data, pd.Series):
    risk_off_data = risk_off_data.to_frame()
print("Risk off assets:")
display(risk_off_data)

## §4 Run WFO's with Stability Analysis

In [ ]:
# §4+§5a — Griglia WFO e pipeline unificata (Momentum + Multifactor)

# Parametri WFO (condivisi tra engines)
ratio                  = "3:1"
metric                 = "Sharpe Ratio"
cores                  = -1
verbose                = False
force_next_year_params = True


# build_wfo_grid() restituisce combinazioni pre-espanse; autoreduce=True
# chiama reduce_grid_via_stability internamente per entrambi gli engine.

ratio_int = int(str(ratio).split(':')[0])
benchmark_start    = benchmark_data.dropna(how='all').index.min()
first_full_year    = pd.Timestamp(f"{benchmark_start.year + 1}-01-01")
pipeline_start_date = first_full_year - pd.DateOffset(years=ratio_int)
autoreduce=True
risk_on_off=True
plot=True

print(f"Benchmark start:         {benchmark_start.date()}")
print(f"Primo anno pieno comune: {first_full_year.date()}")
print(f"Pipeline start:          {pipeline_start_date.date()}")
print(f"Primo OOS atteso:        {first_full_year.date()}")

results_pipeline = {}
for engine in ["Momentum", "Multifactor"]:
    
    grid = build_wfo_grid(engine=engine, profile=profile, asset_type=portfolio.get('asset_type', 'stock'))
    asset_type=portfolio.get('asset_type', 'stock')
    wfo_audit_path_base=str(reports_dir / f"{portfolio_title.replace(' ', '_').lower()}_{year}")

    results_pipeline[engine] = run_wfo_pipeline(
        stocks_data_raw=stocks_data_raw,
        stocks_data=stocks_data,
        benchmark_data=benchmark_data,
        benchmark_data_raw=benchmark_data_raw,
        tickers=tickers,
        risk_off_data=risk_off_data,
        ratio=ratio,
        metric=metric,
        start_date=pipeline_start_date,
        end_date=end_date,
        cores=cores,
        verbose=verbose,
        force_next_year_params=False,
        param_grid=grid,
        engine=engine,
        autoreduce=autoreduce,
        portfolio_title=portfolio_title,
        benchmark_title=benchmark_title,
        init_cash=init_cash,
        risk_on_off=risk_on_off,
        plot=plot,
        profile=profile,
        asset_type=asset_type,
        wfo_audit_path_base=wfo_audit_path_base,
        benchmark_prices=benchmark_data_raw,
    )

# Salva Portfolio object vectorbt + metadati per resume §7
from u_functions import update_latest_symlink
save_wfo_portfolio_objects(
    results_pipeline   = results_pipeline,
    output_dir         = reports_dir,
    ptf_name           = portfolio_title,
    year               = year,
    profilo            = profile,
    pipeline_constants = dict(
        ratio                  = ratio,
        metric                 = metric,
        force_next_year_params = force_next_year_params,
        autoreduce             = autoreduce,
        risk_on_off            = risk_on_off,
        pipeline_start_date    = pipeline_start_date,
    ),
)
if update_latest_symlink(reports_dir):
    print("Symlink 'latest' aggiornato.")

### §4a — Confronto WFO
Confronto metriche di performance degli engines/path disponibili

In [ ]:
metrics_df = compare_wfo_pipelines(
    results         = results_pipeline,
    portfolio_title = portfolio_title,
    benchmark_title = benchmark_title,
    plot_radar      = True,
    save_plots      = True,
    plots_dir       = plots_dir,
)
display(metrics_df)


### §4b — OFC and MC Pipeline

In [ ]:
wfo_runs = run_ofc_mc_pipeline(
    results_pipeline  = results_pipeline,
    profile           = profile,
    portfolio         = portfolio,
    stocks_data       = stocks_data,
    benchmark_data    = benchmark_data,
    benchmark_data_raw= benchmark_data_raw,
    tickers           = tickers,
    init_cash         = init_cash,
    plots_dir         = plots_dir,
)


In [ ]:
metrics_df = compare_wfo_pipelines(
    results         = results_pipeline,
    portfolio_title = portfolio_title,
    benchmark_title = benchmark_title,
    plot_radar      = True,
    save_plots      = True,
    plots_dir       = plots_dir,
)
display(metrics_df)


In [ ]:

# check_v1 = quick_sanity_check(pf_rot_std, pf_rot_std_base, label="Alpha World — Momentum")
# check_v2 = quick_sanity_check(pf_rot_std_v2, pf_rot_std_base_v2, label="Alpha World — Multifactor")

# # Riusabile programmaticamente, es. per skip condizionale:
# if not check_v2['proceed']:
#     print("Salto OFC/MC per v2 (troppi segnali d'allarme)")

In [ ]:
# ── §6 — OFC + MC per ciascun engine ──────────────────────────────────────────
wfo_runs = run_ofc_mc_pipeline(
    results_pipeline  = results_pipeline,
    profile           = profile,
    portfolio         = portfolio,
    stocks_data       = stocks_data,
    benchmark_data    = benchmark_data,
    benchmark_data_raw= benchmark_data_raw,
    tickers           = tickers,
    init_cash         = init_cash,
    plots_dir         = plots_dir,
)

In [ ]:
# for name in ["Momentum", "Multifactor"]:
#     signals = results_pipeline[name]['ofc']['report']['signals']
#     print(f"\n{'='*60}\n  {name} — dettaglio segnali OFC\n{'='*60}")
#     for sig_name, sig_data in signals.items():
#         print(f"  {sig_name}: {sig_data}")

## §5 — Analisi tecnica e relazione

Analisi delle metriche e creazione di una relazione llm

In [ ]:
# pf_rot_momentum = results_pipeline["Momentum"]["pf_rot"]
# stats = pf_rot_momentum.stats()
# # print(stats)
# print("CAGR annualizzato momentun manuale:", (stats["End Value"]/stats["Start Value"])**(365/stats["Period"].days) - 1)
# pf_rot_momentum = results_pipeline["Multifactor"]["pf_rot"]
# stats = pf_rot_momentum.stats()
# # print(stats)
# print("CAGR annualizzato multifactor manuale:", (stats["End Value"]/stats["Start Value"])**(365/stats["Period"].days) - 1)

In [ ]:
# Relazione Tecnica
_report = generate_final_report(
    results_pipeline           = results_pipeline,
    portfolio_title            = portfolio_title,
    year                       = year,
    profile                    = profile,
    benchmark_title            = benchmark_title,
    tickers                    = tickers,
    pipeline_start_date        = pipeline_start_date,
    wfo_file_save              = wfo_file_save,
    survivorship_bias_universe = survivorship_bias_universe,
    reports_dir                = reports_dir,
    plots_dir                  = plots_dir,
    ratio                      = ratio,
    metric                     = metric,
)


## §6 — Scelta PTF 
Sceglie il path da promuovere: engine e variante

In [ ]:
# Salvataggio WFO per il RUNTIME (engine e variante scelti dall'architetto)
#

# Compilare engine_scelto a mano DOPO aver letto la relazione tecnica PDF.
# Valori possibili: le chiavi di results_pipeline (es. "Momentum", "Multifactor")
engine_scelto = "Multifactor"   # <-- compilare a mano dopo lettura PDF

# Compilare variant_scelta a mano DOPO aver letto la relazione tecnica.
# Valori possibili: "RISK_ON_OFF" (default, overlay difensivo attivo
# a runtime) oppure "BASE" (nessun overlay).
# variant_scelta = "RISK_ON_OFF"   # <-- compilare a mano
variant_scelta = "BASE"   # <-- compilare a mano

save_wfo_for_runtime(
    engine_scelto         = engine_scelto,
    results_pipeline      = results_pipeline,
    wfo_runs              = wfo_runs,
    start_date            = start_date,
    end_date              = end_date,
    wfo_file_save         = wfo_file_save,
    metric                = metric,
    ratio                 = ratio,
    force_next_year_params= force_next_year_params,
    variant_scelta        = variant_scelta,
)

### §6a — Performance del PTF prescelto

In [ ]:
out = generate_rotational_portfolio_performance(
    results_pipeline=results_pipeline,
    engine=engine_scelto,
    variant_scelta=variant_scelta,
    portfolio_title=portfolio_title,
    benchmark=benchmark_title,
    benchmark_data=benchmark_data,
    alpha_analysis=True,
    show_plots=True,
    # plot_start_date='2026-05-11'
)

### §6b — Relazione per l'investitore

In [ ]:
_ri_result = generate_relazione_investitore_report(
    out=out,
    portfolio=portfolio,   # un solo parametro, deriva tutto il resto
    reports_dir=reports_dir,
    year=year,
    profile=profile,
)

# print(relazione_investitore_md)

## §7 — Engine Sanity Check

In [ ]:
# Engine Structural Check  (engine_scelto / variant_scelta)
# Controllo strutturale: concentrazione, churn, copertura universo.
# Complementare a OFC/MC — non bloccante, non li sostituisce.
_pf_key  = "pf_rot_base"      if variant_scelta == "BASE" else "pf_rot"
_sel_key = "sel_tickers_base" if variant_scelta == "BASE" else "sel_tickers"

health_df, ticker_df, selection_log, details = build_engine_health_check(
    results_pipeline[engine_scelto][_pf_key],
    results_pipeline[engine_scelto][_sel_key],
    prices     = stocks_data,
    start_date = start_date,
    end_date   = end_date,
    include_prev = True,
)
my_display(health_df,      "Health check motore rotazionale")
my_display(selection_log,  "Audit selezioni")